### Installs & Imports

In [1]:
!pip install --upgrade transformers datasets huggingface_hub
!pip install fsspec==2025.3.0

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.27.0
    Uninstalling huggingface_hub-1.27.0:
      Successfully uninstalled huggingface_hub-1.27.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.15.0
    Uninstalling transformers-5.15.0:
      Successfully uninstalled transforme

### Dataset Loading & Preprocessing

In [2]:
train_path = "train_clean.csv" if os.path.exists("train_clean.csv") else "data/train_clean.csv"
val_path = "val_clean.csv" if os.path.exists("val_clean.csv") else "data/val_clean.csv"
test_path = "test_clean.csv" if os.path.exists("test_clean.csv") else "data/test_clean.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

train_df = train_df.dropna(subset=["clean_prompt"])
val_df = val_df.dropna(subset=["clean_prompt"])
test_df = test_df.dropna(subset=["clean_prompt"])

y_train = train_df["target"].values
y_val = val_df["target"].values
y_test = test_df["target"].values

### Tokenization, Dataset Creation & Hyperparameter Tuning

In [3]:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN_BERT = 128

def encode_bert_texts(texts, max_len):
    return bert_tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

X_train_bert = encode_bert_texts(train_df['clean_prompt'], MAX_LEN_BERT)
X_val_bert = encode_bert_texts(val_df['clean_prompt'], MAX_LEN_BERT)
X_test_bert = encode_bert_texts(test_df['clean_prompt'], MAX_LEN_BERT)

class WildGuardDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = WildGuardDataset(X_train_bert, y_train)
val_dataset = WildGuardDataset(X_val_bert, y_val)
test_dataset = WildGuardDataset(X_test_bert, y_test)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average='macro')
    return {'macro_f1': macro_f1}

bert_configs = [
    {'learning_rate': 2e-5, 'batch_size': 16},
    {'learning_rate': 3e-5, 'batch_size': 32},
    {'learning_rate': 5e-5, 'batch_size': 16}
]

best_bert_f1 = 0
best_bert_model = None
tuning_log = []

for i, config in enumerate(bert_configs):
    config_str = f"lr={config['learning_rate']}, batch_size={config['batch_size']}"

    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

    training_args = TrainingArguments(
        output_dir=f'./results_bert_{i}',
        num_train_epochs=2,
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=config['batch_size'],
        learning_rate=config['learning_rate'],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    eval_results = trainer.evaluate()
    val_macro_f1 = eval_results['eval_macro_f1']

    tuning_log.append(['BERT Base', config_str, val_macro_f1])

    if val_macro_f1 > best_bert_f1:
        best_bert_f1 = val_macro_f1
        best_bert_model = model


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

...

Epoch,Training Loss,Validation Loss,Macro F1
1,0.172873,0.169697,0.942717
2,0.076160,0.151721,0.959056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Training Loss,Validation Loss,Epoch,Macro F1
0.076160,0.151721,2,0.959056


### Hyperparameter Tuning Summary, Test Evaluation & Export

In [4]:
tuning_df = pd.DataFrame(tuning_log, columns=["Model", "Config", "Val F1"])
print("==========================================")
print(" BERT Base Hyperparameter Tuning Summary ")
print("==========================================")
display(tuning_df)

test_trainer = Trainer(model=best_bert_model)
predictions = test_trainer.predict(test_dataset)
logits = predictions.predictions
probs = softmax(logits, axis=1)
test_preds = np.argmax(logits, axis=1)

bert_predictions_df = pd.DataFrame({
    "bert_preds": test_preds,
    "prob_0": probs[:, 0],
    "prob_1": probs[:, 1],
    "prob_2": probs[:, 2],
    "prob_3": probs[:, 3]
})

tuning_df.to_csv("bert_tuning_log.csv", index=False)
bert_predictions_df.to_csv("bert_predictions.csv", index=False)

if os.path.exists("BERT Data"):
    tuning_df.to_csv("BERT Data/bert_tuning_log.csv", index=False)
    bert_predictions_df.to_csv("BERT Data/bert_predictions.csv", index=False)

class_names = ["Benign_Vanilla", "Benign_Adversarial", "Harmful_Vanilla", "Harmful_Adversarial"]

acc = accuracy_score(y_test, test_preds)
macro_f1 = f1_score(y_test, test_preds, average="macro")

print("
==========================================")
print(" BERT Base Final Test Evaluation ")
print("==========================================")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test Macro F1: {macro_f1:.4f}")

print("
Classification Report:")
print(classification_report(y_test, test_preds, target_names=class_names))

cm = confusion_matrix(y_test, test_preds)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("BERT Base Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()